In [1]:
import pandas as pd
import os
from ukhls_variables import VARIABLE_MAP

# --- SETTINGS ---
SIPHER_PKL = "../data/1_pickle_sipher/sipher_optimized.pkl"
BACKFILL_PKL = "../data/3_backfill_ukhls_waves/o_indresp_backfilled.pkl"

# Add generated variables that aren't in the base mapping but get created in intake
EXTENDED_MAP = VARIABLE_MAP.copy()
EXTENDED_MAP['primary_disability'] = 'Type of Impairment (Collapsed)'

def run_backfill_integrity():
    print("--- Integrity Check: Master Backfilled Data ---")
    
    if not os.path.exists(SIPHER_PKL):
        print(f"Error: SIPHER map not found at {SIPHER_PKL}")
        return
        
    if not os.path.exists(BACKFILL_PKL):
        print(f"Error: Backfilled data not found at {BACKFILL_PKL}. Has the backfill pipeline been run?")
        return

    # 1. Load Data
    print("Loading data...")
    df_sipher = pd.read_pickle(SIPHER_PKL)[['pidp']]
    df_master = pd.read_pickle(BACKFILL_PKL)
    
    if 'pidp' not in df_master.columns:
        print("Error: 'pidp' column not found in backfilled master file.")
        return

    total_sipher_rows = len(df_sipher)
    sipher_pid_series = df_sipher['pidp']

    # 2. Overall Match Analytics
    master_ids = set(df_master['pidp'].dropna().unique())
    matched_rows_count = sipher_pid_series.isin(master_ids).sum()

    print("\n" + "="*75)
    print("MATCH SUMMARY")
    print("="*75)
    print(f"Total rows in DataFrame:     {len(df_master):,}")
    print(f"Total rows in SIPHER Map:    {total_sipher_rows:,}")
    print(f"Matched rows with SIPHER:    {matched_rows_count:,}")
    print(f"SIPHER System Coverage:      {(matched_rows_count / total_sipher_rows) * 100:.2f}%\n")
    
    # 3. Variable Fill Analytics
    print("="*75)
    print("VARIABLE FILL RATE (%)")
    print("="*75)
    
    report_rows = []
    
    for base_col, label in EXTENDED_MAP.items():
        if base_col == 'pidp':
            continue
            
        # The backfill script typically drops the wave prefixes outside of pidp for unified columns, 
        # but let's be flexible in case they are "o_" prefixed.
        target_col = None
        if base_col in df_master.columns:
            target_col = base_col
        elif f"o_{base_col}" in df_master.columns:
            target_col = f"o_{base_col}"
        else:
            # Fallback mapping
            matches = [c for c in df_master.columns if c.endswith(f"_{base_col}")]
            if matches:
                target_col = matches[0]

        if target_col:
            null_count = int(df_master[target_col].isnull().sum())
            available_total = len(df_master) - null_count
            fill_rate_total = (available_total / len(df_master)) * 100 if len(df_master) else 0.0
            
            # See how it fills purely within matched SIPHER rows
            valid_pids = set(df_master.loc[df_master[target_col].notnull(), 'pidp'])
            available_in_sipher = int(sipher_pid_series.isin(valid_pids).sum())
            fill_rate_sipher = (available_in_sipher / total_sipher_rows) * 100 if total_sipher_rows else 0.0

            report_rows.append({
                'Feature Description': label,
                'Column': target_col,
                'Total Available': available_total,
                'Total Fill Rate (%)': round(fill_rate_total, 2),
                'SIPHER Matched Available': available_in_sipher,
                'SIPHER Fill Rate (%)': round(fill_rate_sipher, 2)
            })
        else:
            report_rows.append({
                'Feature Description': label,
                'Column': f"Missing ({base_col})",
                'Total Available': 0,
                'Total Fill Rate (%)': 0.0,
                'SIPHER Matched Available': 0,
                'SIPHER Fill Rate (%)': 0.0
            })

    # Sort report by SIPHER fill rate descending to easily see what's well-populated
    report_df = pd.DataFrame(report_rows)
    report_df = report_df.sort_values(by="SIPHER Fill Rate (%)", ascending=False).reset_index(drop=True)
    
    print(report_df.to_string(index=False))

run_backfill_integrity()

--- Integrity Check: Master Backfilled Data ---
Loading data...

MATCH SUMMARY
Total rows in DataFrame:     47,354
Total rows in SIPHER Map:    52,853,971
Matched rows with SIPHER:    52,853,971
SIPHER System Coverage:      100.00%

VARIABLE FILL RATE (%)
                                 Feature Description               Column  Total Available  Total Fill Rate (%)  SIPHER Matched Available  SIPHER Fill Rate (%)
              Derived Age (Current age at interview)             o_age_dv            47329                99.95                  52853971                100.00
                                      Household Size             o_hhsize            47354               100.00                  52853971                100.00
                 Total monthly personal income gross         o_fimngrs_dv            47349                99.99                  52853971                100.00
Employment Status (2=Employed, 5=Retired, 7=Student)             o_jbstat            47280              